# Small Language Model Lab: Did Distillation Help?

## 1. Install dependencies

In [6]:
%pip -q install -U "transformers>=4.45.0" accelerate datasets "pandas==2.2.3" matplotlib
print("If pip reports a package conflict, restart the Colab runtime after installation and rerun from the top.")

If pip reports a package conflict, restart the Colab runtime after installation and rerun from the top.


## 2. Load the models

In [7]:
import re, time, random, numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from IPython.display import display, Markdown

DISTILLED_ID = "AakritiA/qwen2.5-0.5b-sst2-distilled"
BASE_ID = "Qwen/Qwen2.5-0.5B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
print("Device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

_models = {}
_tokenizers = {}

def load_model(model_id):
    if model_id not in _models:
        print(f"Loading {model_id} ...")
        tok = AutoTokenizer.from_pretrained(model_id)
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype=dtype,
            low_cpu_mem_usage=True
        )
        model.to(device)
        model.eval()
        _tokenizers[model_id] = tok
        _models[model_id] = model
        print("Loaded.")
    return _tokenizers[model_id], _models[model_id]

# Load the distilled model first; this is the main model to experiment with.
tokenizer, model = load_model(DISTILLED_ID)
print("Ready:", DISTILLED_ID)

Device: cpu
Loading AakritiA/qwen2.5-0.5b-sst2-distilled ...


config.json:   0%|          | 0.00/1.31k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/697 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/139 [00:00<?, ?B/s]

Loaded.
Ready: AakritiA/qwen2.5-0.5b-sst2-distilled


## 3. Predict sentiment

In [8]:
def predict_sentiment(review, model_id=DISTILLED_ID, max_new_tokens=3):
    tok, mdl = load_model(model_id)
    prompt = f"[label] Review: {review.strip()}\nSentiment:"
    inputs = tok(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    start = time.perf_counter()
    with torch.inference_mode():
        output = mdl.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tok.eos_token_id,
            eos_token_id=tok.eos_token_id
        )

    elapsed = time.perf_counter() - start
    new_tokens = output[0, inputs["input_ids"].shape[1]:]
    raw = tok.decode(new_tokens, skip_special_tokens=True).strip()

    # Accept the label when it is the complete output or the first word.
    match = re.match(r"^\s*(positive|negative)\b", raw, re.IGNORECASE)
    label = match.group(1).capitalize() if match else "Unclear"

    return {
        "label": label,
        "raw_output": raw,
        "latency_seconds": round(elapsed, 3)
    }


examples = [
    "The food was delicious and the service was wonderful.",
    "I waited two hours and the movie was painfully boring.",
    "It was okay, nothing special.",
    "Although the plot was slow, the performances were excellent."
]
for text in examples:
    result = predict_sentiment(text)
    print(f"Review: {text}\\nPrediction: {result['label']} | raw={result['raw_output']!r} | {result['latency_seconds']}s\\n")

Review: The food was delicious and the service was wonderful.\nPrediction: Positive | raw='positive' | 1.267s\n
Review: I waited two hours and the movie was painfully boring.\nPrediction: Negative | raw='negative' | 0.965s\n
Review: It was okay, nothing special.\nPrediction: Positive | raw='positive' | 0.916s\n
Review: Although the plot was slow, the performances were excellent.\nPrediction: Positive | raw='positive' | 0.902s\n


## 4. Try your own review

In [9]:
from IPython.display import display
import ipywidgets as widgets

review_box = widgets.Textarea(
    value="The product arrived early, works perfectly, and exceeded my expectations.",
    description="Review:",
    layout=widgets.Layout(width="100%", height="100px")
)
model_choice = widgets.Dropdown(
    options=[
        ("Distilled student (0.5B)", DISTILLED_ID),
        ("Original student (0.5B)", BASE_ID),
    ],
    value=DISTILLED_ID,
    description="Model:"
)
run_button = widgets.Button(description="Predict sentiment", button_style="primary")
output_box = widgets.Output()

def on_predict_clicked(_):
    with output_box:
        output_box.clear_output()
        result = predict_sentiment(review_box.value, model_choice.value)
        print("Prediction:", result["label"])
        print("Raw model output:", repr(result["raw_output"]))
        print("Generation latency:", result["latency_seconds"], "seconds")

run_button.on_click(on_predict_clicked)
display(model_choice, review_box, run_button, output_box)

Dropdown(description='Model:', options=(('Distilled student (0.5B)', 'AakritiA/qwen2.5-0.5b-sst2-distilled'), …

Textarea(value='The product arrived early, works perfectly, and exceeded my expectations.', description='Revie…

Button(button_style='primary', description='Predict sentiment', style=ButtonStyle())

Output()

## 5. Compare both students on SST-2 validation

In [10]:
# Change to 872 for the complete SST-2 validation split.
N_EVAL = 200
SEED = 42
MAX_NEW_TOKENS = 3

dataset = load_dataset("nyu-mll/glue", "sst2", split="validation")
indices = list(range(len(dataset)))
random.Random(SEED).shuffle(indices)
indices = indices[:min(N_EVAL, len(indices))]
eval_rows = [{"text": dataset[i]["sentence"], "label": "Positive" if int(dataset[i]["label"]) == 1 else "Negative"} for i in indices]
print(f"Evaluating {len(eval_rows)} of {len(dataset)} validation examples.")

README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

sst2/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.11MB            

sst2/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

sst2/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 72.8kB            

sst2/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

sst2/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

Evaluating 200 of 872 validation examples.


In [11]:
def evaluate_model(model_id, rows):
    predictions, raw_outputs, latencies = [], [], []
    for i, row in enumerate(rows, start=1):
        out = predict_sentiment(row["text"], model_id=model_id, max_new_tokens=MAX_NEW_TOKENS)
        predictions.append(out["label"])
        raw_outputs.append(out["raw_output"])
        latencies.append(out["latency_seconds"])
        if i % 25 == 0 or i == len(rows):
            print(f"{model_id}: {i}/{len(rows)}")
    valid = [i for i, p in enumerate(predictions) if p in ("Positive", "Negative")]
    accuracy = (sum(predictions[i] == rows[i]["label"] for i in valid) / len(valid)) if valid else float("nan")
    overall_accuracy = sum(predictions[i] == rows[i]["label"] for i in range(len(rows))) / len(rows)
    return {
        "model": model_id,
        "accuracy_on_parseable": accuracy,
        "accuracy_counting_unclear_as_wrong": overall_accuracy,
        "unclear_rate": sum(p == "Unclear" for p in predictions) / len(predictions),
        "mean_generation_seconds": float(np.mean(latencies)),
        "predictions": predictions,
        "raw_outputs": raw_outputs,
        "latencies": latencies,
    }

base_results = evaluate_model(BASE_ID, eval_rows)
distilled_results = evaluate_model(DISTILLED_ID, eval_rows)

summary = pd.DataFrame([
    {k: v for k, v in r.items() if k not in ("predictions", "raw_outputs", "latencies")}
    for r in [base_results, distilled_results]
])
display(summary.style.format({
    "accuracy_on_parseable": "{:.1%}",
    "accuracy_counting_unclear_as_wrong": "{:.1%}",
    "unclear_rate": "{:.1%}",
    "mean_generation_seconds": "{:.3f}"
}))

Loading Qwen/Qwen2.5-0.5B-Instruct ...


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded.
Qwen/Qwen2.5-0.5B-Instruct: 25/200
Qwen/Qwen2.5-0.5B-Instruct: 50/200
Qwen/Qwen2.5-0.5B-Instruct: 75/200
Qwen/Qwen2.5-0.5B-Instruct: 100/200
Qwen/Qwen2.5-0.5B-Instruct: 125/200
Qwen/Qwen2.5-0.5B-Instruct: 150/200
Qwen/Qwen2.5-0.5B-Instruct: 175/200
Qwen/Qwen2.5-0.5B-Instruct: 200/200
AakritiA/qwen2.5-0.5b-sst2-distilled: 25/200
AakritiA/qwen2.5-0.5b-sst2-distilled: 50/200
AakritiA/qwen2.5-0.5b-sst2-distilled: 75/200
AakritiA/qwen2.5-0.5b-sst2-distilled: 100/200
AakritiA/qwen2.5-0.5b-sst2-distilled: 125/200
AakritiA/qwen2.5-0.5b-sst2-distilled: 150/200
AakritiA/qwen2.5-0.5b-sst2-distilled: 175/200
AakritiA/qwen2.5-0.5b-sst2-distilled: 200/200


,model,accuracy_on_parseable,accuracy_counting_unclear_as_wrong,unclear_rate,mean_generation_seconds
0,Qwen/Qwen2.5-0.5B-Instruct,88.5%,85.0%,4.0%,1.423
1,AakritiA/qwen2.5-0.5b-sst2-distilled,93.0%,93.0%,0.0%,1.178


## 6. Inspect disagreements and mistakes


In [12]:
comparison = pd.DataFrame({
    "text": [r["text"] for r in eval_rows],
    "gold": [r["label"] for r in eval_rows],
    "base_prediction": base_results["predictions"],
    "distilled_prediction": distilled_results["predictions"],
    "base_raw": base_results["raw_outputs"],
    "distilled_raw": distilled_results["raw_outputs"],
})
comparison["base_correct"] = comparison["base_prediction"] == comparison["gold"]
comparison["distilled_correct"] = comparison["distilled_prediction"] == comparison["gold"]

print("Distilled correct, baseline wrong:")
display(comparison[(~comparison["base_correct"]) & comparison["distilled_correct"]].head(15))

print("Baseline correct, distilled wrong:")
display(comparison[comparison["base_correct"] & (~comparison["distilled_correct"])].head(15))

print("Both models wrong or unclear:")
display(comparison[(~comparison["base_correct"]) & (~comparison["distilled_correct"])].head(15))

Distilled correct, baseline wrong:


,text,gold,base_prediction,distilled_prediction,base_raw,distilled_raw,base_correct,distilled_correct
45,"the film is powerful , accessible and funny .",Positive,Unclear,Positive,To determine the,positive,False,True
53,the fly-on-the-wall method used to document ru...,Positive,Unclear,Positive,To determine the,positive,False,True
58,"a difficult , absorbing film that manages to c...",Positive,Unclear,Positive,To express their,positive,False,True
68,not since tom cruise in risky business has an ...,Positive,Negative,Positive,negative\nTo,positive,False,True
70,it gets onto the screen just about as much of ...,Positive,Unclear,Positive,Given the options,positive,False,True
84,"there is nothing outstanding about this film ,...",Positive,Negative,Positive,negative\nTo,positive,False,True
86,true tale of courage -- and complicity -- at a...,Positive,Negative,Positive,negative\nTo,positive,False,True
87,zhang ... has done an amazing job of getting r...,Positive,Unclear,Positive,Given the options,positive,False,True
106,the movie is n't just hilarious : it 's witty ...,Positive,Unclear,Positive,Given the options,positive,False,True
109,"liotta put on 30 pounds for the role , and has...",Positive,Negative,Positive,negative\nWas,positive,False,True


Baseline correct, distilled wrong:


,text,gold,base_prediction,distilled_prediction,base_raw,distilled_raw,base_correct,distilled_correct
38,"the longer the movie goes , the worse it gets ...",Negative,Negative,Positive,negative\nTo,positive,True,False
54,"you wo n't like roger , but you will quickly r...",Negative,Negative,Positive,negative\nThe,positive,True,False
80,i 'll bet the video game is a lot more fun tha...,Negative,Negative,Positive,negative\nTo,positive,True,False
158,"portentous and pretentious , the weight of wat...",Negative,Negative,Positive,negative\n[,positive,True,False
173,every nanosecond of the the new guy reminds yo...,Negative,Negative,Positive,negative\nThis,positive,True,False
185,"the movie , directed by mick jackson , leaves ...",Negative,Negative,Positive,negative\nTo,positive,True,False


Both models wrong or unclear:


,text,gold,base_prediction,distilled_prediction,base_raw,distilled_raw,base_correct,distilled_correct
0,harrison 's flowers puts its heart in the righ...,Positive,Negative,Negative,negative\nTo,negative,False,False
2,a movie that successfully crushes a best selli...,Positive,Negative,Negative,negative\nTo,negative,False,False
13,if your taste runs to ` difficult ' films you ...,Positive,Negative,Negative,negative\nTo,negative,False,False
90,"miller is playing so free with emotions , and ...",Positive,Negative,Negative,negative\nTo,negative,False,False
94,it seems like i have been waiting my whole lif...,Positive,Negative,Negative,negative\nThis,negative,False,False
116,"good film , but very glum .",Positive,Negative,Negative,negative\nWas,negative,False,False
164,morton uses her face and her body language to ...,Positive,Negative,Negative,negative\n[,negative,False,False
184,"despite its title , punch-drunk love is never ...",Positive,Negative,Negative,negative\n[,negative,False,False
